# A/B Testing para LLMs

> Aprende a diseñar y analizar experimentos A/B para comparar modelos,
> prompts y configuraciones LLM con rigor estadístico.

## Objetivos
- Diseñar un experimento A/B con tráfico dividido entre dos variantes
- Calcular significancia estadística con test de proporciones
- Medir métricas de calidad con LLM-as-judge
- Implementar un router de tráfico con porcentajes configurables

## 1. Instalación de dependencias

In [ ]:
%pip install anthropic scipy pandas matplotlib --quiet

## 2. Diseño del experimento A/B

In [ ]:
import anthropic
import random
import json
import time
import hashlib
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

client = anthropic.Anthropic()

# Definir las dos variantes del experimento
VARIANTE_A = {
    "nombre": "A_haiku_directo",
    "modelo": "claude-haiku-4-5-20251001",
    "system_prompt": "Eres un asistente de soporte al cliente. Responde de forma directa y concisa.",
    "max_tokens": 200,
    "temperatura": 0.3,
}

VARIANTE_B = {
    "nombre": "B_haiku_empático",
    "modelo": "claude-haiku-4-5-20251001",
    "system_prompt": "Eres un asistente de soporte al cliente empático y detallado. Reconoce siempre el problema del usuario antes de dar la solución.",
    "max_tokens": 300,
    "temperatura": 0.5,
}

print("=== DISEÑO DEL EXPERIMENTO A/B ===")
print(f"Variante A: {VARIANTE_A['nombre']}")
print(f"  Modelo: {VARIANTE_A['modelo']}")
print(f"  Sistema: {VARIANTE_A['system_prompt'][:60]}...")
print(f"\nVariante B: {VARIANTE_B['nombre']}")
print(f"  Modelo: {VARIANTE_B['modelo']}")
print(f"  Sistema: {VARIANTE_B['system_prompt'][:60]}...")
print("\nMétricas a medir:")
print("  - Satisfacción (LLM-judge 1-10)")
print("  - Resolución del problema (bool)")
print("  - Latencia (segundos)")
print("  - Tokens consumidos")

## 3. Router de tráfico y recopilación de datos

In [ ]:
def asignar_variante(user_id: str, porcentaje_b: float = 0.5) -> str:
    """Asigna variante de forma determinista por user_id (sticky assignment)."""
    hash_val = int(hashlib.md5(user_id.encode()).hexdigest(), 16)
    return "B" if (hash_val % 100) < (porcentaje_b * 100) else "A"

def llamar_variante(variante: dict, pregunta: str) -> dict:
    """Llama a una variante y devuelve respuesta + métricas."""
    inicio = time.perf_counter()
    resp = client.messages.create(
        model=variante["modelo"],
        max_tokens=variante["max_tokens"],
        system=variante["system_prompt"],
        messages=[{"role": "user", "content": pregunta}]
    )
    latencia = time.perf_counter() - inicio
    return {
        "respuesta": resp.content[0].text,
        "tokens_entrada": resp.usage.input_tokens,
        "tokens_salida": resp.usage.output_tokens,
        "latencia_s": latencia,
    }

def evaluar_calidad(pregunta: str, respuesta: str) -> dict:
    """LLM-as-judge evalúa la calidad de la respuesta."""
    prompt = f"""Evalúa la calidad de esta respuesta de soporte al cliente.

Pregunta del cliente: {pregunta}
Respuesta del asistente: {respuesta}

Criterios:
1. ¿Resuelve el problema? (sí/no)
2. ¿Es clara y comprensible? (1-10)
3. ¿El cliente estaría satisfecho? (1-10)

Responde solo con JSON:
{{"resuelve": true/false, "claridad": 0-10, "satisfaccion": 0-10}}"""

    resp = client.messages.create(
        model="claude-haiku-4-5-20251001", max_tokens=100,
        messages=[{"role": "user", "content": prompt}]
    ).content[0].text.strip()
    try:
        if "```" in resp:
            resp = resp.split("```")[1].lstrip("json")
        return json.loads(resp)
    except:
        return {"resuelve": True, "claridad": 7, "satisfaccion": 7}

# Preguntas de soporte simuladas
PREGUNTAS = [
    "No puedo iniciar sesión, me dice que mi contraseña es incorrecta",
    "¿Cómo cancelo mi suscripción sin perder mis datos?",
    "Mi pedido llegó dañado, ¿qué hago?",
    "La aplicación se cierra sola cada vez que abro el chat",
    "¿Puedo cambiar el correo de mi cuenta?",
    "Cobrasteis dos veces en mi tarjeta este mes",
    "No encuentro cómo exportar mis datos en PDF",
    "El soporte telefónico no coge el teléfono",
]

# Simular usuarios y ejecutar experimento
print("Ejecutando experimento A/B...")
registros = []

for i, pregunta in enumerate(PREGUNTAS):
    user_id = f"user_{i:04d}"
    variante_asignada = asignar_variante(user_id)
    variante = VARIANTE_A if variante_asignada == "A" else VARIANTE_B

    resultado = llamar_variante(variante, pregunta)
    calidad = evaluar_calidad(pregunta, resultado["respuesta"])

    registros.append({
        "user_id": user_id,
        "variante": variante_asignada,
        "pregunta": pregunta[:50],
        "latencia_s": resultado["latencia_s"],
        "tokens_totales": resultado["tokens_entrada"] + resultado["tokens_salida"],
        "resuelve": calidad["resuelve"],
        "claridad": calidad["claridad"],
        "satisfaccion": calidad["satisfaccion"],
    })
    print(f"  [{variante_asignada}] {pregunta[:40]}... → satisfacción: {calidad['satisfaccion']}/10")

df = pd.DataFrame(registros)
print("\n✓ Experimento completado")

## 4. Análisis estadístico de resultados

In [ ]:
df_a = df[df["variante"] == "A"]
df_b = df[df["variante"] == "B"]

print("=== RESUMEN POR VARIANTE ===")
resumen = df.groupby("variante").agg(
    n=("user_id", "count"),
    satisfaccion_media=("satisfaccion", "mean"),
    tasa_resolucion=("resuelve", "mean"),
    latencia_media=("latencia_s", "mean"),
    tokens_medio=("tokens_totales", "mean"),
).round(3)
print(resumen.to_string())

# Test estadístico: Mann-Whitney U (no paramétrico) para satisfacción
if len(df_a) > 0 and len(df_b) > 0:
    stat, p_valor = stats.mannwhitneyu(
        df_a["satisfaccion"].values,
        df_b["satisfaccion"].values,
        alternative="two-sided"
    )
    print(f"\n=== TEST ESTADÍSTICO (Mann-Whitney U) ===")
    print(f"  Estadístico U: {stat:.1f}")
    print(f"  p-valor: {p_valor:.4f}")
    print(f"  Significativo (α=0.05): {'SÍ' if p_valor < 0.05 else 'NO (más datos necesarios)'}")

    # Tamaño de efecto de Cohen (diferencia estandarizada)
    media_a = df_a["satisfaccion"].mean()
    media_b = df_b["satisfaccion"].mean()
    std_pool = df["satisfaccion"].std()
    cohen_d = abs(media_b - media_a) / std_pool if std_pool > 0 else 0
    print(f"  Diferencia: B={media_b:.2f} vs A={media_a:.2f}")
    print(f"  Cohen's d (tamaño de efecto): {cohen_d:.3f}")
    interpretacion = "pequeño" if cohen_d < 0.5 else "medio" if cohen_d < 0.8 else "grande"
    print(f"  Interpretación: efecto {interpretacion}")

    ganador = "B" if media_b > media_a else "A"
    print(f"\n→ Variante ganadora provisional: {ganador}")

    # Cálculo de tamaño muestral necesario
    from math import ceil, sqrt
    alpha = 0.05
    potencia = 0.80
    efecto_min = 0.5  # diferencia mínima detectable en puntos
    n_necesario = ceil(16 * (std_pool**2) / (efecto_min**2))  # aproximación
    print(f"\nPara detectar diferencia de {efecto_min} puntos con 80% potencia:")
    print(f"  N necesario por variante: ~{n_necesario} (actualmente: {len(df_a)})")

## 5. Visualización y conclusiones

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle("Resultados del Experimento A/B — LLM Soporte", fontsize=13, fontweight="bold")

colores = {"A": "#3498db", "B": "#2ecc71"}

# Satisfacción por variante
for variante in ["A", "B"]:
    data = df[df["variante"] == variante]["satisfaccion"]
    axes[0].bar(variante, data.mean(), color=colores[variante], alpha=0.85, edgecolor="black")
    axes[0].errorbar(variante, data.mean(), yerr=data.std(), fmt="none", color="black", capsize=8)
axes[0].set_title("Satisfacción Media (±σ)", fontweight="bold")
axes[0].set_ylabel("Puntuación (1-10)")
axes[0].set_ylim(0, 10)
for v in ["A", "B"]:
    m = df[df["variante"] == v]["satisfaccion"].mean()
    axes[0].text(v, m + 0.3, f"{m:.1f}", ha="center", fontweight="bold")

# Tasa de resolución
tasas = df.groupby("variante")["resuelve"].mean()
axes[1].bar(tasas.index, tasas.values * 100,
            color=[colores[v] for v in tasas.index], alpha=0.85, edgecolor="black")
axes[1].set_title("Tasa de Resolución (%)", fontweight="bold")
axes[1].set_ylabel("% de problemas resueltos")
axes[1].set_ylim(0, 100)
for v, tasa in tasas.items():
    axes[1].text(v, tasa * 100 + 2, f"{tasa:.0%}", ha="center", fontweight="bold")

# Latencia vs calidad (scatter)
for variante in ["A", "B"]:
    sub = df[df["variante"] == variante]
    axes[2].scatter(sub["latencia_s"], sub["satisfaccion"],
                   color=colores[variante], label=f"Variante {variante}",
                   s=100, alpha=0.8, edgecolor="black")
axes[2].set_title("Latencia vs Satisfacción", fontweight="bold")
axes[2].set_xlabel("Latencia (s)")
axes[2].set_ylabel("Satisfacción")
axes[2].legend()

plt.tight_layout()
plt.savefig("ab_testing_resultados.png", dpi=150, bbox_inches="tight")
plt.show()
print("\nGráfica guardada en 'ab_testing_resultados.png'")
print("\n=== CONCLUSIÓN ===")
print("Con datos limitados (8 muestras) no se puede establecer significancia estadística.")
print("En producción real: ejecuta el experimento durante 1-2 semanas con cientos de usuarios.")
print("Usa la función asignar_variante() para enrutar tráfico real de forma determinista.")